In [2]:
import carla
import sys
import math
from queue import Queue

CARLA_AGENT_PATH = '/home/oem/workspace/carla_workshop/carla_0.9.14/PythonAPI/carla/'
sys.path.append(CARLA_AGENT_PATH)
from agents.navigation import controller as ctr


class DriveWaypoint(object):
    """
    상태 머신(State Machine) 기반의 안전한 Waypoint 추종 클래스
    """
    def __init__(self, vehicle, pid_controller, world):
        self.car = vehicle
        self.pid = pid_controller
        self.world = world
        self.dir_wp_que = Queue()
        self.is_stopping = False # 차량의 현재 주행 상태를 관리하는 플래그
        
        # '현재 위치'가 아닌 추종해야 할 '목표 웨이포인트'를 별도로 관리
        self.target_waypoint = self.world.get_map().get_waypoint(self.car.get_location())

    def drive_step(self, speed=30.0, debug=False):

        if self.is_stopping:
            control = carla.VehicleControl(throttle=0.0, steer=0.0, brake=1.0)
            self.car.apply_control(control)
            return
        
        current_loc = self.car.get_location()
        
        # [핵심 수정 1] 차량의 현재 실제 속도(m/s)를 계산
        velocity = self.car.get_velocity()
        current_speed_ms = math.sqrt(velocity.x**2 + velocity.y**2 + velocity.z**2)
        
        # [핵심 수정 2] 속도에 비례하는 동적 주시 거리(Look-ahead) 및 갱신 반경 계산
        # 시속 0일때 최소 3m, 속도가 빠를수록 거리가 멀어짐 (예: 시속 72km(20m/s)일 때 약 13m 앞을 봄)
        dynamic_lookahead = max(3.0, current_speed_ms * 0.5 + 3.0) 
        
        # 목표 도달 판정 거리도 속도에 비례해 넓혀줌 (고속일수록 목표점을 넉넉히 스쳐지나가도 도달한 것으로 인정)
        reach_threshold = max(1.5, current_speed_ms * 0.2)

        dist_to_target = current_loc.distance(self.target_waypoint.transform.location)

        if dist_to_target <= reach_threshold: # 목표점과의 거리가 도달
            if not self.dir_wp_que.empty():
                self.target_waypoint = self.dir_wp_que.get()
                print("[시스템] 큐(Queue)에서 새 이벤트 웨이포인트 적용됨")
            else:
                # [핵심 수정 3] 하드코딩된 3.0 대신 동적 거리 적용
                next_wps = self.target_waypoint.next(dynamic_lookahead)
                if next_wps: 
                    self.target_waypoint = next_wps[0]

        if debug:
            self.world.debug.draw_point(
                self.target_waypoint.transform.location, 
                size=0.15, 
                color=carla.Color(255, 0, 0), # 고속 주행 시 목표점이 멀리 찍히는 것을 빨간 점으로 확인
                life_time=0.1
            )

        control = self.pid.run_step(target_speed=speed, waypoint=self.target_waypoint)
        self.car.apply_control(control)

    def add_left_waypoint(self, distance=20):
        """
        현재 위치에서 distance(m) 앞의 좌측 차선 웨이포인트를 큐에 추가
        """
        current_wp = self.world.get_map().get_waypoint(self.car.get_location())
        next_wps = current_wp.next(distance)
        
        if next_wps:
            left_wp = next_wps[0].get_left_lane()
            # [핵심 방어 로직] 좌측 차선이 실제로 존재하는지 검증
            if left_wp is not None:
                self.dir_wp_que.put(left_wp)
                print(f"[명령] {distance}m 앞 좌측 차선 변경 예약 완료")
            else:
                print("[경고] 좌측에 유효한 차선이 없습니다. 차선 변경 명령 무시됨.")

    def add_right_waypoint(self, distance=20):
        """
        현재 위치에서 distance(m) 앞의 우측 차선 웨이포인트를 큐에 추가
        """
        current_wp = self.world.get_map().get_waypoint(self.car.get_location())
        next_wps = current_wp.next(distance)
        
        if next_wps:
            right_wp = next_wps[0].get_right_lane()
            # [핵심 방어 로직] 우측 차선이 실제로 존재하는지 검증
            if right_wp is not None:
                self.dir_wp_que.put(right_wp)
                print(f"[명령] {distance}m 앞 우측 차선 변경 예약 완료")
            else:
                print("[경고] 우측에 유효한 차선이 없습니다. 차선 변경 명령 무시됨.")

    def stop(self):
        self.is_stopping = True
        print("[명령] 차량 정지 명령이 적용되었습니다.")
    
    def resume(self):
        self.is_stopping = False
        print("[명령] 차량 주행 재개 명령이 적용되었습니다.")  
        

In [3]:
import typing
import collections
import numpy as np
import cv2

YOLO_MODEL_PATH = '/home/oem/workspace/carla_workshop/YOLO/runs/detect/train2/weights/best.pt' # coco8 학습 모델 경로

def yolo_setup(yolo_model_path):
    if not hasattr(typing, 'OrderedDict'):
        typing.OrderedDict = collections.OrderedDict

    from ultralytics import YOLO
    model = YOLO(yolo_model_path) # coco8 학습 모델
    print(f"[시스템] YOLO 모델 로드 완료.")
    return model

def _prepr_sensor_img(sensor_img: carla.Image) -> np.ndarray:
    # 이미지 전처리 로직
    img_array = np.frombuffer(sensor_img.raw_data, dtype=np.dtype("uint8")) # 메모리에 직접 접근하여 가져옴
    img_array = np.reshape(img_array, (sensor_img.height, sensor_img.width, 4)) # 1차원 배열 -> (높이, 너비, 채널) 형태로 변환
    img_array_bgr = img_array[:, :, :3] # BGRA -> BGR
    return img_array_bgr

def _visualize_detections(results):
    # 감지 결과 시각화 로직
    annotated_frame = results[0].plot() # YOLO 모델의 첫 번째 결과를 이미지에 그려줌
    cv2.imshow("YOLOv8 CARLA Inference", annotated_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        cv2.destroyAllWindows()

def _vehicle_spawn(world, blueprint: carla.ActorBlueprint, spawn_point: carla.Transform):
    # 차량 액터 소환 로직
    import time
    vc_rt = carla.VehicleControl(reverse=True)
    vc_rf = carla.VehicleControl(reverse=False) 

    actor = world.spawn_actor(blueprint, spawn_point)

    time.sleep(1)
    actor.apply_control(vc_rt)
    time.sleep(0.5)
    actor.apply_control(vc_rf)

    if actor is not None:
        print(f"[시스템] 액터 '{blueprint.id}'가 성공적으로 소환되었습니다.")
    else:
        print(f"[경고] 액터 '{blueprint.id}' 소환에 실패했습니다. 스폰 포인트를 확인하세요.")
    return actor
        

# host_ip='localhost', port=2000
def situation_assessment_and_decision_making_setup(client:carla.Client, delta_sec: float=0.025)-> list:
    # 목표 1 
    # carla 라이브러리 임포트 및 클라이언트 연결, 맵 로드
    actor_list = []
    # client = carla.Client(host_ip, port)

    client.load_world('Town04')

    world = client.get_world()

    # 차량 actor blueprint 정의
    # 주행-전방
    front_bp = world.get_blueprint_library().find('vehicle.mercedes.coupe_2020')
    front_bp.set_attribute('role_name', 'hero')
    front_bp.set_attribute('color','255,255,255')

    # 주행 후방
    rear_bp = world.get_blueprint_library().find('vehicle.mercedes.coupe_2020')
    rear_bp.set_attribute('role_name', 'hero')
    

    # 주행-좌
    left_bp = world.get_blueprint_library().find('vehicle.mercedes.coupe_2020')
    left_bp.set_attribute('role_name', 'hero')

    # 주행-우
    right_bp = world.get_blueprint_library().find('vehicle.mercedes.coupe_2020')
    right_bp.set_attribute('role_name', 'hero')
    

    # 장애물 1
    obs1_bp = world.get_blueprint_library().find('vehicle.citroen.c3')
    obs1_bp.set_attribute('role_name', 'hero')

    # 장애물 2
    obs2_bp = world.get_blueprint_library().find('vehicle.chevrolet.impala')
    obs2_bp.set_attribute('role_name', 'hero')

    # AMB
    amb_bp = world.get_blueprint_library().find('vehicle.ford.ambulance')
    amb_bp.set_attribute('role_name', 'hero')

    # 차량 소환 위치 정의
    spawn_point_front = carla.Transform(carla.Location(x=160.377335, y=14.680666, z=9.333585), carla.Rotation(pitch=0, yaw=-179.57, roll=0)) # 전방
    spawn_point_rear = carla.Transform(carla.Location(x=175.377335, y=14.680666, z=9.333585), carla.Rotation(pitch=0, yaw=-179.57, roll=0)) # 후방
    spawn_point_left = carla.Transform(carla.Location(x=168.311737, y=18.066513, z=9.648239), carla.Rotation(pitch=0, yaw=-179.57, roll=0)) # 좌
    spawn_point_right = carla.Transform(carla.Location(x=165.756546, y=11.099593, z=9.763625), carla.Rotation(pitch=0, yaw=-179.57, roll=0)) # 우
    spawn_point_obs1 = carla.Transform(carla.Location(x=-54.930981, y=13.533400, z=11.055361), carla.Rotation(pitch=-0.808291, yaw=-144.808273, roll=-0.576660)) # 장애물 1
    spawn_point_obs2 = carla.Transform(carla.Location(x=-53.498806, y=10.286665, z=11.085323), carla.Rotation(pitch=-0.896940, yaw=159.454361, roll=0.357310)) # 장애물 2
    spawn_point_amb = carla.Transform(carla.Location(x=-48.693542, y=16.434429, z=11.161311), carla.Rotation(pitch=-1.049438, yaw=-179.041122, roll=-0.013855)) # 앰뷸런스

    # 차량 소환
    front = _vehicle_spawn(world, front_bp, spawn_point_front)
    rear = _vehicle_spawn(world, rear_bp, spawn_point_rear)
    left = _vehicle_spawn(world, left_bp, spawn_point_left)
    right = _vehicle_spawn(world, right_bp, spawn_point_right)
    obs1 = _vehicle_spawn(world, obs1_bp, spawn_point_obs1)
    obs2 = _vehicle_spawn(world, obs2_bp, spawn_point_obs2)
    amb = _vehicle_spawn(world, amb_bp, spawn_point_amb)

    actor_list.extend([front, rear, left, right, obs1, obs2, amb])

    settings = world.get_settings()
    settings.synchronous_mode = True
    settings.fixed_delta_seconds = delta_sec # 100 FPS, 탐지 코드 때문에 틱 레이트를 낮출 경우 오히려 FPS는 낮아지는 현상 확인
    world.apply_settings(settings)

    return  actor_list




In [5]:
# import time
# import typing
# import collections
# import cv2
import carla
import numpy as np
from queue import Queue

YOLO_MODEL_PATH = '/home/oem/workspace/carla_workshop/YOLO/runs/detect/train2/weights/best.pt' # 모델 경로

def run_situation_assessment():

    try:
        client = carla.Client('localhost', 2000)
        world = client.get_world()

        actor_list = situation_assessment_and_decision_making_setup(client)

        # 카메라 센서 설정
        camera_bp = world.get_blueprint_library().filter('sensor.camera.rgb')[0]
        camera_bp.set_attribute('image_size_x', '640') # yolo 모델의 최적 입력 크기에 맞춰 설정
        camera_bp.set_attribute('image_size_y', '640')
        camera_bp.set_attribute('fov', '90')

        # 카메라 센서 위치 설정
        camera_transform = carla.Transform(carla.Location(x=1.5, z=2.4))

        # 카메라 생성 및 ego 차량에 부착
        camera = world.spawn_actor(camera_bp, camera_transform, attach_to=actor_list[0]) # actor_list[0]은 전방 차량으로 가정
        actor_list.append(camera) # 정리 단계에서 카메라도 함께 제거하기 위해 리스트에 추가

        print("\n[시스템] 카메라 생성 및 부착 완료.")

        model = yolo_setup(YOLO_MODEL_PATH) # YOLO 모델 로드


        # 제어기 및 DriveWaypoint 객체 단일 초기화
        # args_lateral_dict = {'K_P': 1.95, 'K_I': 0.05, 'K_D': 0.2, 'dt': 0.05}
        args_lateral_dict = {'K_P': 0.8, 'K_I': 0.05, 'K_D': 0.1, 'dt': 0.05} # 고속 주행 시 안정성을 위한 튜닝
        args_long_dict = {'K_P': 1.0, 'K_I': 0.05, 'K_D': 0.0, 'dt': 0.05}
        
        pid_controller = ctr.VehiclePIDController(actor_list[0], args_lateral_dict, args_long_dict)
        pid_controller_rear = ctr.VehiclePIDController(actor_list[1], args_lateral_dict, args_long_dict)
        pid_controller_left = ctr.VehiclePIDController(actor_list[2], args_lateral_dict, args_long_dict)
        pid_controller_right = ctr.VehiclePIDController(actor_list[3], args_lateral_dict, args_long_dict)
        

        driver_front = DriveWaypoint(actor_list[0], pid_controller, world)
        driver_rear = DriveWaypoint(actor_list[1], pid_controller_rear, world)
        driver_left = DriveWaypoint(actor_list[2], pid_controller_left, world)
        driver_right = DriveWaypoint(actor_list[3], pid_controller_right, world)
        

        # 서버 동기화를 위한 이미지 큐 생성 
        image_queue = Queue()
        camera.listen(lambda image: image_queue.put(image))

        

        # 4. 메인 제어 루프 (약 10초 주행 = 500 틱 * 0.02초)
        for tick in range(700):
            # 4-1. 서버 틱 전진
            world.tick() 

            # time.sleep(0.01)

            image = image_queue.get() # 큐에서 틱과 동기화된 센터 데이터 획득
            bgr_image = _prepr_sensor_img(image) # 이미지 전처리 함수

            # 4-3 YOLOv8 모델로 객체 탐지 수행
            results = model.predict(source=bgr_image, verbose=False)

            # [시나리오 트리거] 전방에 특정 객체가 감지되면 정지 명령 수행
            for result in results:
                boxes = result.boxes
                for box in boxes:
                    cls = int(box.cls[0]) # 탐지 객체의 클래스 ID를 담은 1차원 텐서
                    if cls == 2: # 탐지 결과가 2: car일 경우
                        print(f"\n[Tick {tick}] 이벤트 발생: 전방 차량 감지 - 정지 명령 하달")
                        driver_front.stop()
                        driver_right.stop()
                        driver_left.stop()
                        driver_rear.stop()
                        break

            
            # 단일 스텝 주행 실행 (디버그 라인 렌더링 활성화)
            driver_front.drive_step(speed=70.0, debug=True)
            driver_left.drive_step(speed=70.0, debug=True)
            driver_right.drive_step(speed=70.0, debug=True)
            driver_rear.drive_step(speed=70.0, debug=True)

            # (선택) 시각화를 위한 화면 출력 (속도 저하의 원인이 될 수 있으므로 테스트용으로만 사용)
            annotated_frame = results[0].plot()
            cv2.imshow("YOLOv8 CARLA Inference", annotated_frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                 break


    finally:
        # 5. 환경 초기화 (비정상 종료 시에도 반드시 실행되어야 하는 방어 로직)
        print("\n[시스템] 시나리오 종료. 할당된 Actor 및 환경을 초기화합니다.")
        settings = world.get_settings()
        settings.synchronous_mode = False
        world.apply_settings(settings)
        client.apply_batch([carla.command.DestroyActor(x) for x in actor_list])
        print("[시스템] 정리 완료.")

# 코드 실행
run_situation_assessment()

[시스템] 액터 'vehicle.mercedes.coupe_2020'가 성공적으로 소환되었습니다.
[시스템] 액터 'vehicle.mercedes.coupe_2020'가 성공적으로 소환되었습니다.
[시스템] 액터 'vehicle.mercedes.coupe_2020'가 성공적으로 소환되었습니다.
[시스템] 액터 'vehicle.mercedes.coupe_2020'가 성공적으로 소환되었습니다.
[시스템] 액터 'vehicle.citroen.c3'가 성공적으로 소환되었습니다.
[시스템] 액터 'vehicle.chevrolet.impala'가 성공적으로 소환되었습니다.
[시스템] 액터 'vehicle.ford.ambulance'가 성공적으로 소환되었습니다.

[시스템] 카메라 생성 및 부착 완료.


/home/oem/miniconda3/envs/p3.7/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[시스템] YOLO 모델 로드 완료.


/home/oem/miniconda3/envs/p3.7/lib/python3.7/site-packages/torch/cuda/__init__.py:88: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0
QFontDatabase: Cannot find font directory /home/oem/miniconda3/envs/p3.7/lib/python3.7/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/oem/miniconda3/envs/p3.7/lib/python3.7/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/oem/miniconda3/envs/p3.7/lib/python3.7/site-packages/cv2/qt/fo


[Tick 425] 이벤트 발생: 전방 차량 감지 - 정지 명령 하달
[명령] 차량 정지 명령이 적용되었습니다.
[명령] 차량 정지 명령이 적용되었습니다.
[명령] 차량 정지 명령이 적용되었습니다.
[명령] 차량 정지 명령이 적용되었습니다.

[Tick 429] 이벤트 발생: 전방 차량 감지 - 정지 명령 하달
[명령] 차량 정지 명령이 적용되었습니다.
[명령] 차량 정지 명령이 적용되었습니다.
[명령] 차량 정지 명령이 적용되었습니다.
[명령] 차량 정지 명령이 적용되었습니다.

[Tick 434] 이벤트 발생: 전방 차량 감지 - 정지 명령 하달
[명령] 차량 정지 명령이 적용되었습니다.
[명령] 차량 정지 명령이 적용되었습니다.
[명령] 차량 정지 명령이 적용되었습니다.
[명령] 차량 정지 명령이 적용되었습니다.

[Tick 447] 이벤트 발생: 전방 차량 감지 - 정지 명령 하달
[명령] 차량 정지 명령이 적용되었습니다.
[명령] 차량 정지 명령이 적용되었습니다.
[명령] 차량 정지 명령이 적용되었습니다.
[명령] 차량 정지 명령이 적용되었습니다.

[Tick 448] 이벤트 발생: 전방 차량 감지 - 정지 명령 하달
[명령] 차량 정지 명령이 적용되었습니다.
[명령] 차량 정지 명령이 적용되었습니다.
[명령] 차량 정지 명령이 적용되었습니다.
[명령] 차량 정지 명령이 적용되었습니다.

[Tick 449] 이벤트 발생: 전방 차량 감지 - 정지 명령 하달
[명령] 차량 정지 명령이 적용되었습니다.
[명령] 차량 정지 명령이 적용되었습니다.
[명령] 차량 정지 명령이 적용되었습니다.
[명령] 차량 정지 명령이 적용되었습니다.

[Tick 451] 이벤트 발생: 전방 차량 감지 - 정지 명령 하달
[명령] 차량 정지 명령이 적용되었습니다.
[명령] 차량 정지 명령이 적용되었습니다.
[명령] 차량 정지 명령이 적용되었습니다.
[명령] 차량 정지 명령이 적용되었습니다.

[Tick 453] 이벤트 발생: 전방 차량 감지 - 정지 명령 하달
[명령] 차량 